## NAMA : ALVIN WELVA AL-GHIFARI
## KELAS: VI B PAGI

In [6]:
import os
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout


## 1. KONFIGURASI PATH & PARAMETER
## Pastikan nama folder ini sama persis dengan folder dataset Anda

In [7]:
DATASET_PATH = 'TrashType_Image_Dataset' 
IMG_SIZE = (128, 128)
BATCH_SIZE = 32
EPOCHS = 10

print("Menginisialisasi pemrosesan data...")

Menginisialisasi pemrosesan data...


## 2. PREPROCESSING & AUGMENTASI DATA
## Membuat generator dengan split 80% training dan 20% validation

In [8]:
datagen = ImageDataGenerator(
    rescale=1./255,          # Mengubah nilai piksel menjadi rentang 0-1
    rotation_range=20,       # Memutar gambar acak sampai 20 derajat
    zoom_range=0.2,          # Memperbesar/memperkecil acak
    horizontal_flip=True,    # Membalik gambar secara horizontal
    validation_split=0.2     # Alokasi 20% data untuk validasi (tes internal)
)
train_data = datagen.flow_from_directory(
    DATASET_PATH,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training',
    shuffle=True
)

# Generator untuk Data Validasi (20%)
val_data = datagen.flow_from_directory(
    DATASET_PATH,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation',
    shuffle=False
)

# Menampilkan kelas/label sampah yang terdeteksi
print("\nKelas/Label Sampah yang ditemukan:", list(train_data.class_indices.keys()))
NUM_CLASSES = train_data.num_classes

Found 2024 images belonging to 6 classes.
Found 503 images belonging to 6 classes.

Kelas/Label Sampah yang ditemukan: ['cardboard', 'glass', 'metal', 'paper', 'plastic', 'trash']


## 3. MEMBANGUN ARSITEKTUR CNN MODEL

In [9]:
print("\nMembangun arsitektur model CNN...")
model = Sequential([
    # Blok Konvolusi 1
    Conv2D(32, (3,3), activation='relu', input_shape=(IMG_SIZE[0], IMG_SIZE[1], 3)),
    MaxPooling2D(2,2),
    
    # Blok Konvolusi 2
    Conv2D(64, (3,3), activation='relu'),
    MaxPooling2D(2,2),
    
    # Blok Konvolusi 3 (Opsional, ditambahkan agar model lebih pintar)
    Conv2D(128, (3,3), activation='relu'),
    MaxPooling2D(2,2),
    
    # Mengubah matriks menjadi vektor satu dimensi
    Flatten(),
    
    # Fully Connected Layer (Jaringan Saraf Tiruan)
    Dense(128, activation='relu'),
    Dropout(0.5), # Mencegah overfitting (menghafal gambar)
    
    # Output Layer (Disesuaikan dengan jumlah kategori sampah Anda)
    Dense(NUM_CLASSES, activation='softmax')
])

# Kompilasi Model
model.compile(
    optimizer='adam', 
    loss='categorical_crossentropy', 
    metrics=['accuracy']
)

model.summary() # Menampilkan ringkasan model di terminal


Membangun arsitektur model CNN...


c:\Users\ASUS VIVOBOOK\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 126, 126, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 63, 63, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 61, 61, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 30, 30, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 28, 28, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 14, 14, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 25088)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │     3,211,392 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 6)              │           774 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,305,414 (12.61 MB)

 Trainable params: 3,305,414 (12.61 MB)

 Non-trainable params: 0 (0.00 B)

## 4. PROSES PELATIHAN (MODEL TRAINING)

In [10]:
print(f"\nMemulai pelatihan model selama {EPOCHS} epoch...")
history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=EPOCHS
)


Memulai pelatihan model selama 10 epoch...
Epoch 1/10
64/64 ━━━━━━━━━━━━━━━━━━━━ 29s 409ms/step - accuracy: 0.2910 - loss: 1.6929 - val_accuracy: 0.3936 - val_loss: 1.5493
Epoch 2/10
64/64 ━━━━━━━━━━━━━━━━━━━━ 25s 389ms/step - accuracy: 0.3977 - loss: 1.4856 - val_accuracy: 0.4851 - val_loss: 1.3268
Epoch 3/10
64/64 ━━━━━━━━━━━━━━━━━━━━ 25s 387ms/step - accuracy: 0.4214 - loss: 1.4106 - val_accuracy: 0.4215 - val_loss: 1.3839
Epoch 4/10
64/64 ━━━━━━━━━━━━━━━━━━━━ 25s 389ms/step - accuracy: 0.4704 - loss: 1.3460 - val_accuracy: 0.5129 - val_loss: 1.2897
Epoch 5/10
64/64 ━━━━━━━━━━━━━━━━━━━━ 24s 380ms/step - accuracy: 0.4615 - loss: 1.3213 - val_accuracy: 0.5070 - val_loss: 1.3408
Epoch 6/10
64/64 ━━━━━━━━━━━━━━━━━━━━ 17s 270ms/step - accuracy: 0.4802 - loss: 1.2675 - val_accuracy: 0.4672 - val_loss: 1.3064
Epoch 7/10
64/64 ━━━━━━━━━━━━━━━━━━━━ 18s 280ms/step - accuracy: 0.5109 - loss: 1.2111 - val_accuracy: 0.5845 - val_loss: 1.0859
Epoch 8/10
64/64 ━━━━━━━━━━━━━━━━━━━━ 17s 261ms/step 

## 5. MENYIMPAN MODEL TERLATIH

## Model disimpan agar nanti bisa langsung dipakai tanpa perlu training ulang

In [11]:
MODEL_NAME = 'model_klasifikasi_sampah.h5'
model.save(MODEL_NAME)
print(f"\nPelatihan Selesai! Model sukses disimpan dengan nama: '{MODEL_NAME}'")


Pelatihan Selesai! Model sukses disimpan dengan nama: 'model_klasifikasi_sampah.h5'
